# Multilingual Speaker Diarization + ASR on Indic YouTube

End-to-end pipeline and results. **Runs from committed checkpoints — no GPU, no
API key and no gated model access required.** The only network call is a one-time
fetch of the assignment's `youtube_segments.csv` (the ground truth, not
redistributed here); everything else is read from `checkpoints/`.

| step | what | needs |
|---|---|---|
| 1 | extract 99 clips from YouTube | network (skipped if audio present) |
| 2 | 4 diarizers → DER / JER / speaker count | GPU, *or* committed RTTMs |
| 3 | Saaras v3 per-segment ASR → cpWER / WDER | Sarvam key, *or* committed transcripts |
| 4a | DOVER-Lap fusion | nothing — pure function of Step 2 |
| 4b | script + numeral normalisation | Bedrock key, *or* committed lookup tables |
| 5 | scoring, tables, failure analysis | nothing |

Set `SUBSET` below to run on 10 clips first and confirm the pipeline end to end.

**Ground truth is never a pipeline input.** It is read only by the scoring cells.


In [1]:
#@title Setup — clone, install, configure
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ParvGoyal08/MultilingualASR.git"
CODE = Path("/content/sarvam-assignment") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not (CODE/".git").exists():
        subprocess.run(["git","clone","-q",REPO_URL,str(CODE)], check=True)
    else:
        subprocess.run(["git","-C",str(CODE),"pull","-q","--ff-only"], check=False)
    # Exactly what the checkpoint path imports: pandas/numpy (module level),
    # scipy for the Hungarian assignment in cpWER and DOVER-Lap, rapidfuzz for
    # the alignment, pyannote.metrics for DER/JER. torch and pyannote.audio are
    # only needed to RE-RUN diarization, which reading checkpoints never does.
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "pandas","numpy","scipy","rapidfuzz","pyannote.metrics","jinja2"], check=False)
sys.path.insert(0, str(CODE))

# ---- the only two switches in this notebook -------------------------------
SUBSET   = 0       # 0 = all 99 clips (~2 min from checkpoints).
                   # Set to 10 to verify the whole pipeline in seconds first.
USE_CKPT = True    # read committed model outputs instead of re-running models
# ---------------------------------------------------------------------------

from sarvam_diar.config import Config, StageFlags
from sarvam_diar import data, reference, diarization, evaluation, asr, refinement, translit
from sarvam_diar import text_metrics as tm
import pandas as pd

ROOT = CODE/"checkpoints" if USE_CKPT else Path("local_out/step4_input")
cfg  = Config.create(root=ROOT, work_dir=Path("/tmp/work"))
meta = Config.create(root=CODE/"local_out" if (CODE/"local_out").exists() else ROOT,
                     work_dir=Path("/tmp/work"))
def show(df, precision=4):
    """Render a table. pandas .style needs jinja2; fall back if it is absent."""
    try:
        from IPython.display import display as _d
        _d(df.style.format(precision=precision).hide(axis="index"))
    except Exception:
        print(df.round(precision).to_string(index=False))

print("code root :", CODE)
print("data root :", ROOT, "(committed checkpoints)" if USE_CKPT else "")
print("subset    :", SUBSET or "all 99 clips")

code root : /Users/parvgoyal/Sarvam/Assignment
data root : /Users/parvgoyal/Sarvam/Assignment/checkpoints (committed checkpoints)
subset    : all 99 clips


## 1 — Dataset and ground truth

99 of 100 clips extracted (one failed), 12.26 h, nine Indic scripts, 2–8
speakers. `ClipInput` carries no reference field, so ground truth is structurally
unreachable from Steps 2–4.

In [2]:
clips_all = {c.clip_id: c for c in data.parse_ground_truth(data.load_segments_csv(meta))}
# One of the 100 failed extraction. Restrict to what actually has model output, so
# every count in this notebook refers to the same 99 clips the writeup reports.
CLIPS = {k: v for k, v in clips_all.items() if diarization.is_done(cfg, "fusion", k)}
IDS = sorted(CLIPS)
if SUBSET:
    IDS = IDS[:SUBSET]
REFS = {c: reference.build_reference(CLIPS[c]) for c in IDS}

import collections
ov = sum(r.stats.get("overlap_sec", 0) for r in REFS.values())
tot = sum(CLIPS[c].end_sec - CLIPS[c].start_sec for c in IDS)
print(f"{len(IDS)} of {len(clips_all)} clips   {tot/3600:.2f} h   "
      f"overlap {ov:.0f}s ({ov/tot:.2%})")
print("scripts:", dict(collections.Counter(r.stats.get("lang_script") for r in REFS.values())))
print("speakers/clip:", dict(sorted(collections.Counter(
    r.stats.get("n_speakers") for r in REFS.values()).items())))

14:47:04 | INFO    | sarvam_diar | segments CSV already cached (/Users/parvgoyal/Sarvam/Assignment/local_out/data/youtube_segments.csv)


14:47:04 | INFO    | sarvam_diar | parsed 100 clips, 9940 segments, 2 dropped as malformed, 0 unparsable entries


99 of 100 clips   12.26 h   overlap 3148s (7.13%)
scripts: {'Devanagari': 25, 'Kannada': 9, 'Gujarati': 12, 'Tamil': 10, 'Bengali': 8, 'Oriya': 9, 'Gurmukhi': 7, 'Malayalam': 7, 'Telugu': 12}
speakers/clip: {2: 25, 3: 29, 4: 28, 5: 9, 6: 4, 7: 2, 8: 2}


## 2 — Baseline diarization

Four systems, scored with **collar 0.0 and overlapping speech included**, as the
brief requires. DER components are pooled in seconds; JER and speaker-count
accuracy are per-clip means (JER has no additive decomposition).

In [3]:
MODELS = ["community-1", "pyannote-3.1", "reverb-v2", "diarizen-large"]
def score_diar(models):
    out = []
    for m in models:
        rows = [evaluation.score_clip(REFS[c], diarization.load_hypothesis(cfg, m, c))
                for c in IDS if diarization.is_done(cfg, m, c)]
        if not rows: continue
        g = evaluation.pool(rows)
        out.append(dict(model=m, clips=g["n_clips"], DER=g["der"],
                        miss=g["der_miss_frac"], FA=g["der_fa_frac"],
                        conf=g["der_confusion_frac"], JER=g["jer_mean"],
                        spk_acc=g["speaker_count_accuracy"], spk_MAE=g["speaker_count_mae"]))
    return pd.DataFrame(out).sort_values("DER")
show(score_diar(MODELS), 4)

model,clips,DER,miss,FA,conf,JER,spk_acc,spk_MAE
reverb-v2,99,0.2521,0.0719,0.0681,0.1121,0.3834,0.6061,0.5354
diarizen-large,99,0.2625,0.1143,0.0640,0.0842,0.3660,0.7273,0.3030
community-1,99,0.2634,0.1079,0.0602,0.0954,0.3768,0.7879,0.2323
pyannote-3.1,99,0.2637,0.1079,0.0602,0.0957,0.3785,0.7172,0.3535


## 3 — ASR on the diarized segments

Per-segment: audio is cut at the diarized turns, so speaker attribution is exact
by construction. Saaras returns one timestamp span per request, so long-form word
assignment is not available to it. Adjacent same-speaker turns within 1.0 s are
merged; segments under 0.30 s are skipped, over 29 s are split.

In [4]:
def score_asr(systems):
    out = []
    for s in systems:
        rows = []
        for c in IDS:
            if not asr.is_done(cfg, s, c): continue
            pr = asr.load_pairs(cfg, s, c)
            r = tm.score_transcript(
                {k: v.split() for k, v in reference.speaker_texts(REFS[c]).items()},
                asr.speaker_texts_from_words(pr), reference.word_stream(REFS[c]), pr)
            r["n_hyp"] = len(pr); rows.append(r)
        if not rows: continue
        g = tm.summarise(rows)
        out.append(dict(system=s, clips=g["n_clips"],
                        ratio=sum(r["n_hyp"] for r in rows)/g["n_ref_words"],
                        WER=g["wer"], cpWER=g["cpwer"], WDER=g["wder"],
                        attribution=g["cp_minus_di_cp"]))
    return pd.DataFrame(out)

BASE, FUS = "sarvam-saaras-v3@reverb-v2", "sarvam-saaras-v3@fusion"
show(score_asr([BASE, FUS]), 4)

system,clips,ratio,WER,cpWER,WDER,attribution
sarvam-saaras-v3@reverb-v2,99,0.9595,0.2728,0.3957,0.1128,0.1229
sarvam-saaras-v3@fusion,99,0.9786,0.2787,0.3181,0.0628,0.0394


## 4a — Step 4: DOVER-Lap fusion

Hungarian label alignment onto a growing centroid, then per-frame voting with
each speaker thresholded **independently** so overlap survives. An argmax vote
cannot emit two simultaneous speakers, and 7.1% of this corpus is overlapped.

GT-free: reads only the three constituent RTTMs. Configuration (equal weights,
2-of-3, `min_dur` 0.20) was selected on the dev half.

In [5]:
FUSION_MODELS = ["community-1", "reverb-v2", "diarizen-large"]
# The fusion regenerates bit-for-bit from its constituents -- verify, do not assume.
same = tried = 0
for c in IDS:
    if not all(diarization.is_done(cfg, m, c) for m in FUSION_MODELS): continue
    tried += 1
    dur = CLIPS[c].end_sec - CLIPS[c].start_sec
    got = refinement.dover_lap({m: diarization.load_hypothesis(cfg, m, c)
                                for m in FUSION_MODELS}, dur, threshold=0.5, min_dur=0.20)
    same += asr.segmentation_key(got) == asr.segmentation_key(
        diarization.load_hypothesis(cfg, "fusion", c))
print(f"fusion reproduces bit-for-bit from its constituents: {same}/{tried} clips")
show(score_diar(FUSION_MODELS + ["fusion"]), 4)

fusion reproduces bit-for-bit from its constituents: 99/99 clips


model,clips,DER,miss,FA,conf,JER,spk_acc,spk_MAE
fusion,99,0.2442,0.1138,0.0498,0.0806,0.3608,0.7980,0.2121
reverb-v2,99,0.2521,0.0719,0.0681,0.1121,0.3834,0.6061,0.5354
diarizen-large,99,0.2625,0.1143,0.0640,0.0842,0.3660,0.7273,0.3030
community-1,99,0.2634,0.1079,0.0602,0.0954,0.3768,0.7879,0.2323


**Read this table carefully.** The fusion beats `community-1`,
`pyannote-3.1` and `diarizen-large` by ~7% relative DER with 80+/99 per-clip
wins. Against `reverb-v2` the DER difference is **not significant**
(Δ −0.0079, 95% CI [−0.0353, +0.0161], 42/57 per clip). What the fusion does win
on `reverb-v2` is JER, speaker-count accuracy (79.8% vs 60.6%) and confusion —
and, decisively, the downstream ASR result below.

## 4b — Step 4: script and numeral normalisation

The reference writes code-switched English **phonetically in the native script**
and numbers as **spoken words**. Saaras writes Latin and digits. Those are
correctly-recognised words scored as substitutions — 3,481 of 21,357.

The stage extracts the Latin/digit vocabulary **from the hypothesis**, has an LLM
render each in the clip's language, and applies a frozen lookup table. It can
only rewrite tokens already emitted in Latin or as digits, so its downside is
bounded at zero.

In [6]:
import json as _json
lat = _json.loads((ROOT/"results"/"translit_table.json").read_text())
num = _json.loads((ROOT/"results"/"numeral_table.json").read_text())
print(f"lookup tables: {sum(len(v) for v in lat.values()):,} Latin, "
      f"{sum(len(v) for v in num.values()):,} numeral entries")
for sc in ("Telugu", "Devanagari", "Gujarati"):
    ex = list(lat.get(sc, {}).items())[:3] + list(num.get(sc, {}).items())[:2]
    print(f"  {sc:<11}" + "   ".join(f"{k}->{v}" for k, v in ex))

XL = "sarvam-saaras-v3@fusion+xlit+num"
show(score_asr([BASE, FUS, XL]), 4)

lookup tables: 1,651 Latin, 382 numeral entries
  Telugu     a->ఎ   about->అబౌట్   also->ఆల్సో   00->సున్న సున్న   000->సున్న సున్న సున్న
  Devanagari a->अ   aahe->आहे   aahet->आहेत   0->शून्य   000->शून्य शून्य शून्य
  Gujarati   a->અ   also->ઓલ્સો   am->એમ   0->શૂન્ય   000->શૂન્ય શૂન્ય શૂન્ય


system,clips,ratio,WER,cpWER,WDER,attribution
sarvam-saaras-v3@reverb-v2,99,0.9595,0.2728,0.3957,0.1128,0.1229
sarvam-saaras-v3@fusion,99,0.9786,0.2787,0.3181,0.0628,0.0394
sarvam-saaras-v3@fusion+xlit+num,99,0.9807,0.2585,0.3017,0.0602,0.0432


## 5 — Held-out evaluation

`results/split.json` is a frozen 50/49 dev/test split, stratified by script and
speaker-count band. Dev and test differ in difficulty, so only **deltas** transfer
across them.

Two estimands appear below and they are not interchangeable. **Pooled** cpWER
(total errors ÷ total reference words) is the headline number and weights long
clips more. The **per-clip mean** delta weights every clip equally and is what the
paired bootstrap resamples, so the CI belongs to that column, not to the pooled
one. Quoting a pooled figure next to a CI computed on means is a real error, so
both are shown side by side.

In [7]:
import random
sp = _json.loads((ROOT/"results"/"split.json").read_text())

def per_clip(system, ids):
    out = {}
    for c in ids:
        if not asr.is_done(cfg, system, c) or c not in REFS: continue
        pr = asr.load_pairs(cfg, system, c)
        out[c] = tm.score_transcript(
            {k: v.split() for k, v in reference.speaker_texts(REFS[c]).items()},
            asr.speaker_texts_from_words(pr), reference.word_stream(REFS[c]), pr)
    return out

rows = []
for name in ("dev", "test", "all"):
    ids = [c for c in IDS if name == "all" or c in set(sp[name])]
    b, a = per_clip(FUS, ids), per_clip(XL, ids)
    common = sorted(set(b) & set(a))
    if not common: continue
    # POOLED -- the headline estimand: total errors / total reference words.
    pool_b, pool_a = (tm.summarise([x[c] for c in common])["cpwer"] for x in (b, a))
    # PER-CLIP MEAN -- the estimand the paired bootstrap over clips applies to.
    d = [a[c]["cpwer"] - b[c]["cpwer"] for c in common]
    rng = random.Random(20260822)
    m = sorted(sum(rng.choice(d) for _ in d)/len(d) for _ in range(4000))
    rows.append({"split": name, "clips": len(common),
                 "pooled_before": pool_b, "pooled_after": pool_a,
                 "pooled_delta": pool_a - pool_b,
                 "mean_delta": sum(d)/len(d), "ci_lo": m[100], "ci_hi": m[3900],
                 "better": sum(x < -1e-9 for x in d), "worse": sum(x > 1e-9 for x in d)})
show(pd.DataFrame(rows), 4)

split,clips,pooled_before,pooled_after,pooled_delta,mean_delta,ci_lo,ci_hi,better,worse
dev,50,0.2676,0.2582,-0.0094,-0.0096,-0.0136,-0.0062,34,0
test,49,0.3552,0.3337,-0.0216,-0.0167,-0.0274,-0.0080,38,1
all,99,0.3181,0.3017,-0.0164,-0.0131,-0.0189,-0.0085,72,1


## 6 — Where the error still is

Overlap is the dominant unfixed weakness: overlapped words are deleted at ~4×
the clean rate, and 52.5% of diarization miss lies in overlapped regions.

Three interventions were built and measured against it end-to-end — a 1-of-3
overlap vote, ConvTasNet source separation, and LLM contextual refinement.
**None improved the system.** Each had a stated hypothesis, a control and a
pre-committed abandon criterion; details and numbers are in `WRITEUP.md` §9 and
`obs.txt` [50]–[52]. Two further ideas, an MSDD-style verifier and an
OSD∩constituent intersection, were dropped after a candidate-precision audit
(11.5% and 23.6%) without being implemented, and are reported as audits.

In [8]:
import numpy as np
R = 0.01
rows = []
for c in IDS:
    if not asr.is_done(cfg, XL, c): continue
    r = REFS[c]; n = int(r.uem[1]/R)
    rl = sorted({t.speaker for t in r.turns}); ri = {l: k for k, l in enumerate(rl)}
    M = np.zeros((len(rl), n), bool)
    for t in r.turns:
        a, b = int(t.start/R), min(n, int(t.end/R))
        if b > a: M[ri[t.speaker], a:b] = True
    ovl = M.sum(0) >= 2
    ws = []
    for u in r.utterances:
        tk = u.text_norm.split()
        if not tk: continue
        st = (u.end - u.start)/len(tk)
        for k, w in enumerate(tk):
            j = int((u.start + (k+0.5)*st)/R)
            ws.append((w, bool(ovl[j]) if 0 <= j < n else False))
    hy = [w for w, _ in asr.load_pairs(cfg, XL, c)]
    for op, i, _ in tm.align([w for w, _ in ws], hy):
        if i < 0: continue
        rows.append((("overlap" if ws[i][1] else "clean"), op))
df = pd.DataFrame(rows, columns=["region", "op"])
t = pd.crosstab(df.region, df.op, normalize="index")
t = (t[["equal", "replace", "delete"]] * 100).rename(
    columns={"equal": "hit %", "replace": "sub %", "delete": "del %"})
# reset_index so the region labels survive -- they ARE the result here
show(t.reset_index(), 1)
print(f"\noverlapped words are deleted {t.loc['overlap','del %']/t.loc['clean','del %']:.1f}x "
      f"as often as clean ones")

region,hit %,sub %,del %
clean,81.7,13.7,4.6
overlap,57.1,24.4,18.4



overlapped words are deleted 4.0x as often as clean ones


## 7 — Final system

```
sarvam-saaras-v3 @ DOVER-Lap(community-1, reverb-v2, diarizen-large)
                 + script/numeral normalisation
```

| | baseline | final | |
|---|---|---|---|
| **cpWER** | 0.3957 | **0.3017** | **−23.8%** |
| **WDER** | 0.1128 | **0.0602** | **−46.6%** |
| WER | 0.2728 | 0.2585 | −5.2% |
| DER | 0.2521 | 0.2442 | not significant |
| JER | 0.3834 | 0.3608 | −5.9% |
| speaker-count acc | 60.6% | 79.8% | +19.2 pt |

Against the oracle floor of cpWER **0.1242** — what a perfect transcript on a
perfect diarization still scores, because one transcript cannot carry two
simultaneous speakers — the final system sits 0.1775 above what is reachable.

Per-video tables: `results/step2_metrics.csv` (495 rows), `results/step3_metrics.csv`
(539 rows). Full analysis: `WRITEUP.md`. Lab notebook: `obs.txt`.